In [ ]:
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score

In [ ]:
INPUT_FODLER = "collected_outputs"
OUTPUT_FOLDER = "regr_models"
PLOT_HIST = False

# MODELS_NAMES = [
#     "cam3_revB_batch1_11_margin_15_rq_wb_alif_clahe_box_01_dfl_01_2nd",
#     "cam3_revB_batch1_11_margin_15_rq_wb_box_01_dfl_01_2nd",
# ]
# MODELS_NAMES = [
#     "cam3_revB_batch1_11_margin_15_rq_yolov8n_wb_alif_clahe_box_01_dfl_01_2nd",
#     "cam3_revB_batch1_11_margin_15_yolov8n_wb_alif_box_01_dfl_01_2nd",
# ]
MODELS_NAMES = [
    "cam3_revB_batch1_12_margin_15_wb_alif_clahe_box_01_dfl_01_2nd",
]

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
def plot_hists(buf_fp32_scores, buf_uint8_scores, buf_fp32_boxes, buf_uint8_boxes):
    scores_dif = np.abs(buf_fp32_scores - buf_uint8_scores).flatten()
    plt.hist(scores_dif, bins=100);
    print(f"Mean={scores_dif.mean():.3f}, Std={scores_dif.std():.3f}, Median={np.median(scores_dif):.3f}, Max={scores_dif.max():.3f}")

    coord_dif = np.abs(buf_fp32_boxes - buf_uint8_boxes)
    print("Coord dif mean:", coord_dif.mean(axis=0))
    print("Coord dif std:", coord_dif.std(axis=0))
    print("Coord dif median:", np.median(coord_dif, axis=0))
    print("Coord dif max:", coord_dif.max(axis=0))

    plt.figure(figsize=(15, 8))
    plt.subplot(3, 2, 1)
    plt.hist(coord_dif[:, 0].flatten(), bins=100)
    plt.title('dx histogram')
    plt.subplot(3, 2, 2)
    plt.hist(coord_dif[:, 1].flatten(), bins=100)
    plt.title('dy histogram')
    plt.subplot(3, 2, 3)
    plt.hist(coord_dif[:, 2].flatten(), bins=100)
    plt.title('dw histogram')
    plt.subplot(3, 2, 4)
    plt.hist(coord_dif[:, 3].flatten(), bins=100)
    plt.title('dh histogram')
    plt.subplot(3, 2, 5)
    plt.hist(coord_dif[:, 4].flatten(), bins=100)
    plt.title('dx_coord histogram')
    plt.subplot(3, 2, 6)
    plt.hist(coord_dif[:, 5].flatten(), bins=100)
    plt.title('dy_coord histogram')

    plt.tight_layout()


In [ ]:
def train_regr(buf_fp32_boxes, buf_uint8_boxes, regr_pth):
    # Train linear regression models for all 6 box coordinates
    coord_names = ['dx', 'dy', 'dw', 'dh', 'dx_coord', 'dy_coord']
    models = []

    params = {}

    # First: Train models on full dataset
    for i in range(6):
        # X = buf_uint8_boxes[:, i].reshape(-1, 1)
        X = buf_uint8_boxes
        y = buf_fp32_boxes[:, i]
        
        # model = LinearRegression(fit_intercept=True)
        model = RidgeCV(fit_intercept=True, alphas=np.logspace(-2, 1, 50))
        model.fit(X, y)
        models.append(model)

        print(f"{coord_names[i]}:")
        print(f"  Coef: {model.coef_}, Intercept: {model.intercept_:.6f}, Alpha: {model.alpha_ if hasattr(model, 'alpha_') else 'N/A'}")
        params[coord_names[i]] = {
            'coef': [float(i) for i in model.coef_],
            'intercept': float(model.intercept_),
            'alpha': float(model.alpha_) if hasattr(model, 'alpha_') else None
        }

    print("\n" + "="*50)
    print("Model training completed\n")

    # MSE comparison: FP32 vs UINT8 (without regression)
    print("\n" + "="*50)
    print("MSE: FP32 vs UINT8 (direct conversion)\n")
    mse_uint8 = mean_squared_error(buf_fp32_boxes, buf_uint8_boxes, multioutput='raw_values')
    for i, name in enumerate(coord_names):
        print(f"{name}: {mse_uint8[i]:.4f}")
    print(f"Overall MSE: {mse_uint8.mean():.4f}")

    # MSE comparison: FP32 vs Regression predictions (UINT8 as input)
    print("\n" + "="*50)
    print("MSE: FP32 vs Regression predictions (UINT8 input)\n")
    predictions = np.column_stack([models[i].predict(buf_uint8_boxes) for i in range(6)])
    mse_regression = mean_squared_error(buf_fp32_boxes, predictions, multioutput='raw_values')
    for i, name in enumerate(coord_names):
        print(f"{name}: {mse_regression[i]:.4f}")
    print(f"Overall MSE: {mse_regression.mean():.4f}")

    # Improvement comparison
    print("\n" + "="*50)
    print("MSE Improvement (UINT8 -> Regression)\n")
    for i, name in enumerate(coord_names):
        improvement = ((mse_uint8[i] - mse_regression[i]) / mse_uint8[i]) * 100
        print(f"{name}: {improvement:.2f}%")
    print(f"Overall: {((mse_uint8.mean() - mse_regression.mean()) / mse_uint8.mean()) * 100:.2f}%")

    file_name = 'wb_alif_clahe_ridge_regression_params_tiny.json'
    with open(regr_pth, 'w') as f:
        json.dump(params, f, indent=4)
    print(f"\nRegression parameters saved to '{regr_pth}'")

In [ ]:
for model_name in MODELS_NAMES:
    print(f"\n{'#'*20} Processing model: {model_name} {'#'*20}\n")

    buf_fp32_scores = np.load(os.path.join(INPUT_FODLER, model_name, 'fp32_scores.npy'))
    buf_fp32_boxes = np.load(os.path.join(INPUT_FODLER, model_name, 'fp32_boxes.npy'))
    buf_uint8_scores = np.load(os.path.join(INPUT_FODLER, model_name, 'uint8_scores.npy'))
    buf_uint8_boxes = np.load(os.path.join(INPUT_FODLER, model_name, 'uint8_boxes.npy'))

    print(buf_fp32_scores.shape, buf_fp32_boxes.shape, buf_uint8_scores.shape, buf_uint8_boxes.shape)

    regr_pth = os.path.join(OUTPUT_FOLDER, f'{model_name}_ridge_regression_params.json')

    if PLOT_HIST:
        plot_hists(buf_fp32_scores, buf_uint8_scores, buf_fp32_boxes, buf_uint8_boxes)

    train_regr(buf_fp32_boxes, buf_uint8_boxes, regr_pth)